In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm

from models.timefm import TimesFMModel
from models.lstm import LSTMModel, QLSTMModel
from models.gan import QGanModel, MultiSequenceGAN
from sklearn.preprocessing import StandardScaler

stocks = ['coalindia', 'tcs', 'hindalco', 'jswsteel']

/home/ton618wsl/qwhether_forecasting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:


# ============================================================
# Configuration
# ============================================================

HISTORICAL_LOOKUP = 30
HORIZON = 15
LAG = 0

LATENT_SIZE = 8
N_QUBITS = 6
QUANTUM_LAYERS = 4
HIDDEN_SIZE = 32

EPOCHS = 200
BATCH_SIZE = 32
LEARNING_RATE = 1e-4

DIVERSITY_LOSS_WEIGHT = 0.8
VARIETY_LOSS_WEIGHT = 1.0

TRAIN_RATIO = 0.8
SEED = 42


# ============================================================
# Load ALL stocks into a dictionary
#
# IMPORTANT:
# Do NOT concatenate the raw time series.
#
# The MultiSequenceGAN receives:
#
# {
#     "AAPL": array(...),
#     "MSFT": array(...),
#     "GOOGL": array(...),
#     ...
# }
#
# The model should create context/target windows independently
# for every stock and only pool the resulting windows.
# ============================================================

data = {
    stock: np.asarray(
        np.load(f"data/{stock}_l10y.npy")
    ).reshape(-1)
    for stock in stocks
}


# ============================================================
# Dataset information
# ============================================================

data_info = []

for stock, series in data.items():

    data_info.append({
        "Stock": stock,
        "N": len(series),
        "Mean": np.mean(series),
        "Std": np.std(series),
        "Variance": np.var(series),
        "Min": np.min(series),
        "Max": np.max(series),
    })


data_info_df = pd.DataFrame(data_info)


print("\n")
print("=" * 110)
print("INPUT DATA STATISTICS")
print("=" * 110)

print(
    data_info_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# Combined raw-data statistics
# ============================================================

all_raw_values = np.concatenate(
    list(data.values())
)

print("\n")
print("=" * 110)
print("COMBINED DATA INFORMATION")
print("=" * 110)

print(
    pd.DataFrame([{
        "Stocks": len(data),
        "Total Observations": len(all_raw_values),
        "Mean": np.mean(all_raw_values),
        "Std": np.std(all_raw_values),
        "Variance": np.var(all_raw_values),
        "Min": np.min(all_raw_values),
        "Max": np.max(all_raw_values),
    }]).to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# Model
# ============================================================

print("\n")
print("=" * 110)
print("INITIALIZING MULTI-STOCK MultiSequenceGAN-QC")
print("=" * 110)

print(f"Stocks            : {list(data.keys())}")
print(f"Historical lookup : {HISTORICAL_LOOKUP}")
print(f"Horizon           : {HORIZON}")
print(f"Lag               : {LAG}")
print(f"Train ratio       : {TRAIN_RATIO}")
print(f"Latent size       : {LATENT_SIZE}")
print(f"Qubits            : {N_QUBITS}")
print(f"Quantum layers    : {QUANTUM_LAYERS}")


msgan_qc = MultiSequenceGAN(
    data=data,
    type="QC",

    historical_lookup=HISTORICAL_LOOKUP,
    horizon=HORIZON,
    lag=LAG,

    latent_size=LATENT_SIZE,
    n_qubits=N_QUBITS,
    quantum_layers=QUANTUM_LAYERS,

    hidden_size=HIDDEN_SIZE,

    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,

    diversity_loss_weight=DIVERSITY_LOSS_WEIGHT,
    variety_loss_weight=VARIETY_LOSS_WEIGHT,

    train_ratio=TRAIN_RATIO,
    seed=SEED,
)


# ============================================================
# Training
# ============================================================

print("\n")
print("=" * 110)
print("TRAINING SINGLE MODEL ON ALL STOCKS")
print("=" * 110)

history = msgan_qc.train()


# ============================================================
# Backtest
# ============================================================

print("\n")
print("=" * 110)
print("RUNNING MULTI-STOCK BACKTEST")
print("=" * 110)

d = msgan_qc.backtest(
    return_all_sequences=True
)


contexts = d["contexts"]
predictions = d["predictions"]
actuals = d["actuals"]
all_seq = d["all_sequences"]


# Optional stock IDs.
#
# If your updated MultiSequenceGAN returns stock information
# from backtest(), this will allow per-stock evaluation.
#
# Expected key:
#     "stock_ids"
#
# If it does not exist, the script simply performs aggregate
# evaluation.

stock_ids = d.get("stock_ids", None)


# ============================================================
# Basic shapes
# ============================================================

print("\n")
print("=" * 110)
print("BACKTEST OUTPUT")
print("=" * 110)

print("Contexts    :", contexts.shape)
print("Predictions :", predictions.shape)
print("Actuals     :", actuals.shape)
print("All seq     :", all_seq.shape)

if stock_ids is not None:
    print("Stock IDs   :", np.asarray(stock_ids).shape)


# ============================================================
# Metrics
# ============================================================

print("\n")
print("=" * 110)
print("CALCULATING METRICS")
print("=" * 110)


prob_result = msgan_qc.probabilistic_metrics(
    all_seq,
    actuals
)


result = msgan_qc.metrics(
    predictions,
    actuals
)


# ============================================================
# Aggregate statistics
# ============================================================

actual_flat = actuals.reshape(-1)
pred_flat = predictions.reshape(-1)

original_mean = np.mean(all_raw_values)
original_std = np.std(all_raw_values)
original_var = np.var(all_raw_values)

actual_mean = np.mean(actual_flat)
actual_std = np.std(actual_flat)
actual_var = np.var(actual_flat)

pred_mean = np.mean(pred_flat)
pred_std = np.std(pred_flat)
pred_var = np.var(pred_flat)


# ============================================================
# Distribution preservation
# ============================================================

variance_ratio = (
    pred_var /
    (actual_var + 1e-12)
)


variance_error_pct = (
    abs(pred_var - actual_var) /
    (actual_var + 1e-12)
    * 100
)


mean_error = pred_mean - actual_mean

std_error = pred_std - actual_std


mean_error_pct = (
    abs(mean_error) /
    (abs(actual_mean) + 1e-12)
    * 100
)


std_error_pct = (
    abs(std_error) /
    (actual_std + 1e-12)
    * 100
)


# ============================================================
# Correlation
# ============================================================

correlation = np.corrcoef(
    actual_flat,
    pred_flat
)[0, 1]


# ============================================================
# Directional accuracy
#
# IMPORTANT:
# This is calculated on the flattened predictions exactly
# like the original script.
# ============================================================

actual_diff = np.diff(actual_flat)
pred_diff = np.diff(pred_flat)

directional_accuracy = (
    np.mean(
        np.sign(actual_diff)
        ==
        np.sign(pred_diff)
    )
    * 100
)


# ============================================================
# Standard forecasting metrics
# ============================================================

overall = result["overall"]

mae = overall.get(
    "mae",
    np.nan
)

rmse = overall.get(
    "rmse",
    np.nan
)

mape = overall.get(
    "mape",
    np.nan
)

r2 = overall.get(
    "r2",
    np.nan
)


# ============================================================
# Aggregate result
# ============================================================

aggregate_result = {

    "Number of Stocks":
        len(data),

    "Total Observations":
        len(all_raw_values),

    "Test Samples":
        len(actuals),

    "Horizon":
        HORIZON,

    "Lag":
        LAG,

    # Original data
    "Original Mean":
        original_mean,

    "Original Std":
        original_std,

    "Original Variance":
        original_var,

    # Actual test data
    "Actual Mean":
        actual_mean,

    "Actual Std":
        actual_std,

    "Actual Variance":
        actual_var,

    # Predictions
    "Pred Mean":
        pred_mean,

    "Pred Std":
        pred_std,

    "Pred Variance":
        pred_var,

    # Distribution
    "Variance Ratio":
        variance_ratio,

    "Variance Error %":
        variance_error_pct,

    "Mean Error":
        mean_error,

    "Mean Error %":
        mean_error_pct,

    "Std Error":
        std_error,

    "Std Error %":
        std_error_pct,

    # Forecasting
    "MAE":
        mae,

    "RMSE":
        rmse,

    "MAPE":
        mape,

    "R2":
        r2,

    # Dependence
    "Correlation":
        correlation,

    "Directional Accuracy %":
        directional_accuracy,
}


# ============================================================
# Aggregate results table
# ============================================================

aggregate_df = pd.DataFrame(
    [aggregate_result]
)


print("\n")
print("=" * 110)
print("AGGREGATE MULTI-STOCK RESULTS")
print("=" * 110)

print(
    aggregate_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# Training history
# ============================================================

print("\n")
print("=" * 110)
print("TRAINING HISTORY")
print("=" * 110)

if isinstance(history, dict):

    print(
        "Available history keys:",
        list(history.keys())
    )


# ============================================================
# 1. TRAINING LOSSES
# ============================================================

if isinstance(history, dict):

    for key, values in history.items():

        if not isinstance(values, (list, np.ndarray)):
            continue

        values = np.asarray(values)

        if values.ndim != 1:
            continue

        if len(values) == 0:
            continue

        plt.figure(figsize=(12, 5))

        plt.plot(
            values,
            label=key
        )

        plt.title(
            f"Multi-Stock MultiSequenceGAN-QC — {key}"
        )

        plt.xlabel("Epoch")
        plt.ylabel(key)

        plt.grid(alpha=0.25)
        plt.legend()

        plt.tight_layout()
        plt.show()


# ============================================================
# 2. DAY +1 FORECAST
# ============================================================

day1_actual = actuals[:, 0]
day1_pred = predictions[:, 0]


plt.figure(figsize=(14, 6))

plt.plot(
    day1_actual,
    label="Actual Day +1"
)

plt.plot(
    day1_pred,
    label="Predicted Day +1",
    linestyle="--"
)

plt.title(
    "Multi-Stock MultiSequenceGAN-QC — Day +1 Forecast"
)

plt.xlabel("Test Window")
plt.ylabel("Value")

plt.grid(alpha=0.25)
plt.legend()

plt.tight_layout()
plt.show()


# ============================================================
# 3. DAY +1 SCATTER
# ============================================================

plt.figure(figsize=(7, 7))

plt.scatter(
    day1_actual,
    day1_pred,
    alpha=0.5
)

min_value = min(
    day1_actual.min(),
    day1_pred.min()
)

max_value = max(
    day1_actual.max(),
    day1_pred.max()
)

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.title(
    "Day +1 — Actual vs Predicted"
)

plt.xlabel("Actual")
plt.ylabel("Predicted")

plt.grid(alpha=0.25)

plt.tight_layout()
plt.show()


# ============================================================
# 4. FULL HORIZON ERROR BY FORECAST STEP
# ============================================================

horizon_mae = np.mean(
    np.abs(
        predictions - actuals
    ),
    axis=0
)

horizon_rmse = np.sqrt(
    np.mean(
        (predictions - actuals) ** 2,
        axis=0
    )
)


plt.figure(figsize=(12, 5))

plt.plot(
    np.arange(1, HORIZON + 1),
    horizon_mae,
    marker="o",
    label="MAE"
)

plt.plot(
    np.arange(1, HORIZON + 1),
    horizon_rmse,
    marker="x",
    label="RMSE"
)

plt.title(
    "Forecast Error by Horizon"
)

plt.xlabel("Forecast Step")
plt.ylabel("Error")

plt.grid(alpha=0.25)
plt.legend()

plt.tight_layout()
plt.show()


# ============================================================
# 5. FULL HORIZON BIAS
# ============================================================

horizon_bias = np.mean(
    predictions - actuals,
    axis=0
)


plt.figure(figsize=(12, 5))

plt.bar(
    np.arange(1, HORIZON + 1),
    horizon_bias
)

plt.axhline(
    0,
    linestyle="--"
)

plt.title(
    "Forecast Bias by Horizon"
)

plt.xlabel("Forecast Step")
plt.ylabel("Mean Prediction Error")

plt.grid(alpha=0.25)

plt.tight_layout()
plt.show()


# ============================================================
# 6. ERROR DISTRIBUTION
# ============================================================

errors = (
    predictions - actuals
).reshape(-1)


plt.figure(figsize=(12, 5))

plt.hist(
    errors,
    bins=50,
    alpha=0.75
)

plt.axvline(
    0,
    linestyle="--"
)

plt.title(
    "Prediction Error Distribution"
)

plt.xlabel("Prediction Error")
plt.ylabel("Frequency")

plt.grid(alpha=0.25)

plt.tight_layout()
plt.show()


# ============================================================
# 7. ACTUAL VS PREDICTED DISTRIBUTION
# ============================================================

plt.figure(figsize=(12, 5))

plt.hist(
    actual_flat,
    bins=50,
    alpha=0.5,
    label="Actual"
)

plt.hist(
    pred_flat,
    bins=50,
    alpha=0.5,
    label="Predicted"
)

plt.title(
    "Actual vs Predicted Distribution"
)

plt.xlabel("Value")
plt.ylabel("Frequency")

plt.grid(alpha=0.25)
plt.legend()

plt.tight_layout()
plt.show()


# ============================================================
# 8. MULTIPLE FORECAST WINDOWS
# ============================================================

n_examples = min(
    6,
    len(actuals)
)


fig, axes = plt.subplots(
    n_examples,
    1,
    figsize=(12, 4 * n_examples),
    squeeze=False
)

axes = axes.flatten()


for idx in range(n_examples):

    ax = axes[idx]

    ensemble = all_seq[idx]

    x_context = np.arange(
        -HISTORICAL_LOOKUP,
        0
    )

    x_future = np.arange(
        0,
        HORIZON
    )

    ax.plot(
        x_context,
        contexts[idx],
        label="Context"
    )

    ax.plot(
        x_future,
        actuals[idx],
        label="Actual",
        marker="o"
    )

    ax.plot(
        x_future,
        predictions[idx],
        label="Prediction",
        linestyle="--",
        marker="x"
    )

    # Plot generated trajectories
    if ensemble.ndim == 2:

        for s in range(
            ensemble.shape[1]
        ):

            ax.plot(
                x_future,
                ensemble[:, s],
                alpha=0.05
            )

    ax.axvline(
        0,
        linestyle=":"
    )

    ax.set_title(
        f"Forecast Window {idx}"
    )

    ax.set_xlabel(
        "Relative Time"
    )

    ax.set_ylabel(
        "Value"
    )

    ax.grid(alpha=0.25)
    ax.legend()


plt.tight_layout()
plt.show()


# ============================================================
# 9. ENSEMBLE MEAN / SPREAD
# ============================================================

if all_seq.ndim == 3:

    ensemble_mean = np.mean(
        all_seq,
        axis=2
    )

    ensemble_std = np.std(
        all_seq,
        axis=2
    )

    # Average uncertainty over test windows

    mean_prediction = np.mean(
        ensemble_mean,
        axis=0
    )

    mean_uncertainty = np.mean(
        ensemble_std,
        axis=0
    )

    mean_actual = np.mean(
        actuals,
        axis=0
    )

    plt.figure(figsize=(12, 6))

    horizon_axis = np.arange(
        1,
        HORIZON + 1
    )

    plt.plot(
        horizon_axis,
        mean_actual,
        label="Actual mean"
    )

    plt.plot(
        horizon_axis,
        mean_prediction,
        label="Generated mean",
        linestyle="--"
    )

    plt.fill_between(
        horizon_axis,
        mean_prediction - mean_uncertainty,
        mean_prediction + mean_uncertainty,
        alpha=0.2,
        label="±1 std"
    )

    plt.title(
        "Average Forecast Trajectory and Ensemble Uncertainty"
    )

    plt.xlabel("Forecast Step")
    plt.ylabel("Value")

    plt.grid(alpha=0.25)
    plt.legend()

    plt.tight_layout()
    plt.show()


# ============================================================
# 10. PER-STOCK RESULTS
#
# This section is activated if backtest() returns stock_ids.
# ============================================================

if stock_ids is not None:

    stock_ids = np.asarray(
        stock_ids
    )

    print("\n")
    print("=" * 110)
    print("PER-STOCK TEST RESULTS")
    print("=" * 110)

    per_stock_results = []

    for stock in np.unique(stock_ids):

        mask = (
            stock_ids == stock
        )

        stock_actual = actuals[mask]
        stock_pred = predictions[mask]

        if len(stock_actual) == 0:
            continue

        stock_actual_flat = (
            stock_actual.reshape(-1)
        )

        stock_pred_flat = (
            stock_pred.reshape(-1)
        )

        stock_mae = mean_absolute_error(
            stock_actual_flat,
            stock_pred_flat
        )

        stock_rmse = np.sqrt(
            mean_squared_error(
                stock_actual_flat,
                stock_pred_flat
            )
        )

        try:
            stock_r2 = r2_score(
                stock_actual_flat,
                stock_pred_flat
            )
        except Exception:
            stock_r2 = np.nan

        try:
            stock_corr = np.corrcoef(
                stock_actual_flat,
                stock_pred_flat
            )[0, 1]
        except Exception:
            stock_corr = np.nan

        stock_actual_diff = np.diff(
            stock_actual_flat
        )

        stock_pred_diff = np.diff(
            stock_pred_flat
        )

        stock_directional_accuracy = (
            np.mean(
                np.sign(stock_actual_diff)
                ==
                np.sign(stock_pred_diff)
            )
            * 100
        )

        per_stock_results.append({

            "Stock": stock,

            "Samples": len(
                stock_actual
            ),

            "MAE": stock_mae,

            "RMSE": stock_rmse,

            "R2": stock_r2,

            "Correlation": stock_corr,

            "Directional Accuracy %":
                stock_directional_accuracy,

            "Actual Mean":
                np.mean(stock_actual_flat),

            "Pred Mean":
                np.mean(stock_pred_flat),

            "Actual Std":
                np.std(stock_actual_flat),

            "Pred Std":
                np.std(stock_pred_flat),

            "Actual Variance":
                np.var(stock_actual_flat),

            "Pred Variance":
                np.var(stock_pred_flat),

        })


    per_stock_df = pd.DataFrame(
        per_stock_results
    )

    print(
        per_stock_df.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )


    # ========================================================
    # PER-STOCK METRIC PLOTS
    # ========================================================

    metric_columns = [
        "MAE",
        "RMSE",
        "R2",
        "Correlation",
        "Directional Accuracy %",
    ]


    for metric in metric_columns:

        plt.figure(figsize=(12, 5))

        plt.bar(
            per_stock_df["Stock"],
            per_stock_df[metric]
        )

        plt.title(
            f"{metric} by Stock"
        )

        plt.xlabel("Stock")
        plt.ylabel(metric)

        plt.xticks(
            rotation=45,
            ha="right"
        )

        plt.grid(
            axis="y",
            alpha=0.25
        )

        plt.tight_layout()
        plt.show()


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n")
print("=" * 120)
print("FINAL MULTI-STOCK MultiSequenceGAN — QC RESULTS")
print("=" * 120)


final_cols = [
    "Number of Stocks",
    "Total Observations",
    "Test Samples",
    "Horizon",
    "Lag",
    "Original Variance",
    "Actual Variance",
    "Pred Variance",
    "Variance Ratio",
    "Variance Error %",
    "MAE",
    "RMSE",
    "MAPE",
    "R2",
    "Correlation",
    "Directional Accuracy %",
]


print(
    aggregate_df[final_cols].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)



INPUT DATA STATISTICS
    Stock    N        Mean        Std      Variance         Min         Max
coalindia 2474  270.970048 101.659925  10334.740451  110.550003  540.400024
      tcs 2474 2663.678111 972.887776 946510.624811 1050.574951 4553.750000
 hindalco 2474  380.253435 213.659964  45650.580306   84.900002 1024.050049
 jswsteel 2474  547.029681 320.635331 102807.015607  125.945000 1280.599976


COMBINED DATA INFORMATION
 Stocks  Total Observations       Mean         Std       Variance       Min         Max
      4                9896 965.482819 1116.816308 1247278.664993 84.900002 4553.750000


INITIALIZING MULTI-STOCK MultiSequenceGAN-QC
Stocks            : ['coalindia', 'tcs', 'hindalco', 'jswsteel']
Historical lookup : 30
Horizon           : 15
Lag               : 0
Train ratio       : 0.8
Latent size       : 8
Qubits            : 6
Quantum layers    : 4

MULTI-STOCK DATASET
Number of stocks : 4
Historical lookup: 30
Forecast horizon : 15
Lag              : 0
Train samples  

KeyboardInterrupt: 